# 머신러닝 기초와 문제정의

1. 규칙을 구성하는 것과 학습하는 것의 차이

> **노트북 사용전 확인**
>
> 셀을 순서대로 실행하지 않으면 변수상태가 꼬일 수 있다.
> 위에서 아래 순서대로 실행하면서 확인하자.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _style import setup
from _ml import build_dataset, FEATURES, SEED

setup()
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)

print("준비완료")

[폰트] Malgun Gothic / unicode_minus=False
준비완료


---
## 1. 코딩과의 차이점

지금까지 해온 프로그래밍은 **규칙을 사람이 정하는 것**이다.

```python
if 거래량 > 1000000 and 등락률 > 5:
    return "급등주"
```

규칙이 명확하면 코딩이 무조건 좋다. 빠르고, 정확하고, 설명이 가능하다.

다만, **규칙을 사람이 알 수 없을 때**는 다르다.

> "오늘 주가가 오를 종목의 조건은 무엇인가?"

거래량, 등락률, 이동평균, 변동성... 수십개 이상의 변수로 인해 if문 작성이 어렵다.
**머신러닝은 데이터에서 그 규칙을 컴퓨터 찾아내게 하는 방법**이다.

In [7]:
#직접 구현해보자

# 정제및 파생변수 생성을 마친 데이터
df = build_dataset()

# 규칙 : 거래량이 평소보다 2배이상 많고, 20일 평균보다 3%이상 높으면 오른다.
rule = (df["volume_ratio"] > 2.0) & (df["ma20_ratio"] > 1.03)

# 이 규칙이 예측한것과 실제 정답을 비교
# df["target_up"] -> 올랐으면 1, 아니면 0
hit = (rule == df["target_up"]).mean()

print(f" 규칙에 해당하는 날 : {rule.sum():,}건 / {len(df):,}건")
print(f" 규칙의 적중률 : {hit:.4f}")
print()
print(f"전체 상승비율 : {df['target_up'].mean()}")


 규칙에 해당하는 날 : 731건 / 87,480건
 규칙의 적중률 : 0.5099

전체 상승비율 : 0.48954046639231824


> **적중률이 매우매우 낮게 나온다**
>
> 사람이 느낌으로 만든 규칙이기 때문에, 어떤 조건을 어떤 값으로 잡아야하는지 어려움이 있다.
> 조건을 바꿔가며 손으로 찾을 수도 있지만, 변수가 7개만 돼도 조합이 폭발적으로 많다.

- 프로그래밍 : 데이터 + 규칙(사람)을 입력해서 결과를 받는다.
- 머신러닝 : 데이터 + 정답을 입력해서 규칙을 만든다.

> 규칙이 명확하면 프로그램밍을 통해 답을 구하면되고, 
> **규칙을 알 수 없을 때** 머신러닝을 이용해서 규칙을 만들면 된다.

---
## 분류와 회귀 - 무엇을 예측하려고 하는가?

**머신러닝에서는 이 판단이 매우 중요하다.**
| | 분류 | 회귀 |
|---|---|---|
|예측가능한 것|범주|수치|
|질문|오를까?내릴까?|얼마나오를까?|
|출력|up/down|2.7%|
|평가지표|정확도, 정밀도, 재현율|MAE,RMSE...|

**같은 데이터로 두 문제를 다 만들 수 있다.**

In [ ]:
# 우리 데이터에는 두가지 정답이 전부있음
# target_ret : 오늘의 수익률 - 수치 -> 회귀용
# target_up : 올랐니? (1/0) - 범주 -> 분류용

sample = df[["code", "date", "target_ret", "target_up"]].head(8)
print(sample.to_string(index=False))

print()
print("같은 행이지만 정답을 두가지로 볼 수 있다.")

 code       date  target_ret  target_up
G0001 2023-10-24    0.015574          1
G0001 2023-10-25   -0.008639          0
G0001 2023-10-26   -0.002960          0
G0001 2023-10-27    0.002676          1
G0001 2023-10-30    0.003128          1
G0001 2023-10-31    0.046732          1
G0001 2023-11-01   -0.001827          0
G0001 2023-11-02    0.011819          1


> **평가 지표를 섞어서 사용할 수 없다**
>
> 분류모델같은 경우 MAE,RMSE...을 사용하거나, 회귀모델에 정확도를 쓸 수 없다.
> "정확도 92%"라는 말은 **분류에서만** 의미가 있다.
> 회귀에서 "정확하게 맞혔다"는 개념자체가 성립하지 않는다.

### 기준선 -> 사람이 정한다.

연속값을 범주로 변경하려면 어디서 자를것이가?를 정해야한다 -> **정답의 기준이된다**

In [ ]:
#기준선을 바꿔가며 '상승'의 비율이 어떻게 변하는지 보자.

print(f"{'기준선':>10}{'양성비율':>12}{'양성건수':>12}")
print("-"*60)

for th in [0.0, 0.01, 0.02, 0.03, 0.05]:
    positive = (df["target_ret"] > th)
    print(f"{th:>9.0%}{positive.mean():>12.2%}{positive.sum():>12,}")